### **Mount Drive and Create Folders**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.makedirs("/content/drive/MyDrive/project/data", exist_ok=True)
os.makedirs("/content/drive/MyDrive/project/tableau", exist_ok=True)
os.makedirs("/content/drive/MyDrive/project/models", exist_ok=True)

Mounted at /content/drive


### **Start Spark**

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Model_Training") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


### **Load Train and Test Data**

In [4]:
train = spark.read.parquet(
"/content/drive/MyDrive/project/data/train.parquet"
)

test = spark.read.parquet(
"/content/drive/MyDrive/project/data/test.parquet"
)

### **Import ML Libraries**

In [5]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.classification import LinearSVC

from pyspark.ml.evaluation import BinaryClassificationEvaluator

### **Create Evaluator**

In [6]:
evaluator = BinaryClassificationEvaluator(
labelCol="label",
rawPredictionCol="rawPrediction",
metricName="areaUnderROC"
)

### **Train Logistic Regression**

In [7]:
lr = LogisticRegression(
featuresCol="scaledFeatures",
labelCol="label",
maxIter=10
)

lr_model = lr.fit(train)

lr_pred = lr_model.transform(test)

lr_auc = evaluator.evaluate(lr_pred)

print("Logistic Regression AUC:", lr_auc)

Logistic Regression AUC: 0.9989071675268314


### **Train Random Forest**

In [17]:
train_sample = train.sample(fraction=0.001, seed=42)
test_sample = test.sample(fraction=0.001, seed=42)

In [20]:
from pyspark.ml.feature import UnivariateFeatureSelector

selector = UnivariateFeatureSelector(
    featuresCol="scaledFeatures",
    outputCol="selectedFeatures",
    labelCol="label"
)

selector.setFeatureType("continuous")
selector.setLabelType("categorical")
selector.setSelectionMode("numTopFeatures")
selector.setSelectionThreshold(500)

selector_model = selector.fit(train_sample)

train_sample = selector_model.transform(train_sample)
test_sample = selector_model.transform(test_sample)

In [22]:
rf = RandomForestClassifier(
    featuresCol="selectedFeatures",
    labelCol="label",
    numTrees=5,
    maxDepth=5,
    maxBins=32
)

rf_model = rf.fit(train_sample)

rf_pred = rf_model.transform(test_sample)

rf_auc = evaluator.evaluate(rf_pred)

print("Random Forest AUC:", rf_auc)

Random Forest AUC: 0.9917457392906496


### **Train Decision Tree**

In [25]:
dt = DecisionTreeClassifier(
  featuresCol="selectedFeatures",
  labelCol="label"
)

dt_model = dt.fit(train_sample)

dt_pred = dt_model.transform(test_sample)

dt_auc = evaluator.evaluate(dt_pred)

print("Decision Tree AUC:", dt_auc)

Decision Tree AUC: 0.9868724090280975


### **Train SVM**

In [26]:
svm = LinearSVC(
featuresCol="selectedFeatures",
labelCol="label",
maxIter=10
)

svm_model = svm.fit(train_sample)

svm_pred = svm_model.transform(test_sample)

svm_auc = evaluator.evaluate(svm_pred)

print("SVM AUC:", svm_auc)

SVM AUC: 0.9941777982496558


### **Save Model Comparison CSV**

In [27]:
import pandas as pd

model_results = pd.DataFrame({

"Model": [

"Logistic Regression",
"Decision Tree",
"Random Forest",
"SVM"

],

"AUC": [

lr_auc,
dt_auc,
rf_auc,
svm_auc

]

})

model_results.to_csv(

"/content/drive/MyDrive/project/tableau/model_comparison.csv",

index=False

)

model_results

,Model,AUC
0,Logistic Regression,0.998907
1,Decision Tree,0.986872
2,Random Forest,0.991746
3,SVM,0.994178


### **Save Feature Importance CSV**

In [28]:
importance = rf_model.featureImportances.toArray()

importance_df = pd.DataFrame({

"feature_index": list(range(len(importance))),
"importance": importance

})

importance_df.to_csv(

"/content/drive/MyDrive/project/tableau/feature_importance.csv",

index=False

)

### **Save Models**

In [29]:
lr_model.write().overwrite().save(
"/content/drive/MyDrive/project/models/logistic_regression")

selector_model.write().overwrite().save(
"/content/drive/MyDrive/project/models/feature_selector"
)

dt_model.write().overwrite().save(
"/content/drive/MyDrive/project/models/decision_tree")

rf_model.write().overwrite().save(
"/content/drive/MyDrive/project/models/random_forest")

svm_model.write().overwrite().save(
"/content/drive/MyDrive/project/models/svm")